In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

In [2]:
from ast import literal_eval

df = pd.read_csv('data/matches_processed.csv')
df["Player"] = df["Player"].apply(literal_eval)

In [3]:
import torch
from torch.utils.data import Dataset
from sklearn.preprocessing import LabelEncoder

class WinPredictionDataset(Dataset):
    def __init__(self, players, result):
        self.players = players
        self.results = torch.tensor(result, dtype=torch.float32)
        
        # Flatten all tokens to build vocabulary
        all_tokens = [item for row in players for subsublist in row for item in subsublist]
        self.tokenizer = LabelEncoder()
        self.tokenizer.fit(all_tokens)  # Fit on all possible tokens
        
        # Pre-encode all data during init (more efficient)
        self.encoded_players = [
            [
                self.tokenizer.transform(subsublist) 
                for subsublist in row
            ] 
            for row in players
        ]
        
    def __len__(self):
        return len(self.players)

    def __getitem__(self, idx):
        return {
            "players": torch.tensor(np.array(self.encoded_players[idx]), dtype=torch.long),  # Shape: [10, 5]
            "results": self.results[idx]  # Shape: [1]
        }

In [4]:
import torch
import torch.nn as nn

class WinPredictionModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, encoder_dim=64):
        super().__init__()
        
        # 1. Embedding Layer 
        self.embedding = nn.Embedding(vocab_size + 1, embed_dim, padding_idx=0)
        
        # 2. 2D CNN (Spatial Feature Extraction)
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=(3,3), padding=1),  # [batch, 32, 10, 5]
            nn.BatchNorm2d(16),
            nn.Dropout(0.2),
            nn.ReLU(),
            nn.MaxPool2d((2,2)),  # [batch, 32, 5, 2]
        )
        
        # 3. Encoder (Processes CNN outputs)
        self.encoder = nn.TransformerEncoder(
            encoder_layer=nn.TransformerEncoderLayer(
                d_model=16*2,  # Flattened CNN channels
                nhead=8,
                dim_feedforward=encoder_dim,
                dropout=0.2,
                batch_first=True
            ),
            num_layers=1,
            enable_nested_tensor=True 
        )
        
        # 4. Classifier
        self.classifier = nn.Sequential(
            nn.Linear(16*2*5, 64),  # 5 players after pooling
            nn.Sigmoid(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # x shape: [batch_size, 10, 5]
        
        # 1. Embed tokens
        x = self.embedding(x)  # [batch, 10, 5, embed_dim]
        
        # 2. Prepare for CNN (average embeddings)
        x = x.mean(dim=-1, keepdim=True)  # [batch, 10, 5, 1]
        x = x.permute(0, 3, 1, 2)  # [batch, 1, 10, 5]
        
        # 3. 2D CNN
        cnn_out = self.cnn(x)  # [batch, 32, 5, 2]
        
        # 4. Prepare for encoder
        batch_size, channels, h, w = cnn_out.shape
        cnn_flat = cnn_out.reshape(batch_size, h, channels*w)  # [batch, 5, 64]
        
        # 5. Transformer encoder
        encoded = self.encoder(cnn_flat)  # [batch, 5, 64]
        
        # 6. Classifier
        out = self.classifier(encoded.reshape(batch_size, -1))
        return torch.sigmoid(out)

In [5]:
from sklearn.metrics import confusion_matrix, accuracy_score

def eval_model(model, val_loader, criterion, device):
    model.eval()
    
    all_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in val_loader:
            players = batch["players"].to(device)
            labels = batch["results"].to(device)
            
            outputs = model(players)
            loss = criterion(outputs, labels.unsqueeze(1)) 
            all_loss += loss.item()
            
            preds = np.round(outputs.cpu().numpy())
            
            all_preds.extend(preds.flatten().tolist())
            all_labels.extend(labels.cpu().numpy().flatten().tolist())
            
    average_loss = all_loss / len(val_loader)            
    accuracy = accuracy_score(all_labels, all_preds)
    conf_matrix = confusion_matrix(all_labels, all_preds)  # Call the function from sklearn.metrics
    return average_loss, accuracy, conf_matrix

In [7]:
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold
import torch
import numpy as np
import visualize

torch.manual_seed(42)
np.random.seed(42)

n_folds = 5
num_epochs = 20
learning_rate = 2e-4
batch_size = 128

dataset = WinPredictionDataset(df["Player"].values, df["Win"].to_numpy())

kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

fold_train_losses = []
fold_val_losses = []
fold_accuracies = []
fold_conf_matrices = []

for fold, (train_idx, val_idx) in enumerate(kf.split(dataset)):
    print(f"Fold {fold + 1}/{kf.n_splits}")
    
    trainset = torch.utils.data.Subset(dataset, train_idx)
    valset = torch.utils.data.Subset(dataset, val_idx)

    train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(valset, batch_size=batch_size, shuffle=False)
    
    # Initialize model, optimizer, and loss function
    model = WinPredictionModel(vocab_size=dataset.tokenizer.classes_.size).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.BCELoss()

    train_losses = []
    val_losses = []
    accuracies = []
    conf_matrices = []

    for epoch in range(num_epochs):
        train_loss = 0
        model.train()
        for batch in train_loader:
            players = batch["players"].to(device)
            results = batch["results"].to(device)
            
            optimizer.zero_grad()
            outputs = model(players)
            loss = criterion(outputs, results.unsqueeze(1))  # Ensure results are the same shape as outputs
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
        average_train_loss = train_loss / len(train_loader)
        average_val_loss, accuracy, conf_matrix = eval_model(model, val_loader, criterion, device)
        
        print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {average_train_loss}/{average_val_loss}, Validation accuracy: {accuracy}")
        
        train_losses.append(average_train_loss)
        val_losses.append(average_val_loss)
        accuracies.append(accuracy)
        
    fold_train_losses.append(train_losses)
    fold_val_losses.append(val_losses)
    fold_accuracies.append(accuracies)
    fold_conf_matrices.append(conf_matrix)
    
# Average the results across folds
train_losses = np.mean(fold_train_losses, axis=0)
val_losses = np.mean(fold_val_losses, axis=0)
accuracies = np.mean(fold_accuracies, axis=0)
conf_matrices = np.mean(fold_conf_matrices, axis=0)

visualize.loss(train_losses, val_loss=val_losses, title=f"Train and Validation Losses across {n_folds} folds")
visualize.accuracy(accuracies, title=f"Validation Accuracy across {n_folds} folds")
visualize.confusion_matrix(conf_matrices, title=f"Confusion Matrix across {n_folds} folds")

Using device: cuda
Fold 1/5
Epoch 1/20, Loss: 0.6914114620950487/0.69452203810215, Validation accuracy: 0.506
Epoch 2/20, Loss: 0.689528038577428/0.6937263794243336, Validation accuracy: 0.507
Epoch 3/20, Loss: 0.6885780550184704/0.6924630030989647, Validation accuracy: 0.5125
Epoch 4/20, Loss: 0.6871288693140424/0.6920378506183624, Validation accuracy: 0.515
Epoch 5/20, Loss: 0.6862436012616233/0.6924123391509056, Validation accuracy: 0.5195
Epoch 6/20, Loss: 0.6847326916361612/0.6925192512571812, Validation accuracy: 0.5235
Epoch 7/20, Loss: 0.6838149873037187/0.6920172311365604, Validation accuracy: 0.521
Epoch 8/20, Loss: 0.683722198009491/0.6927406676113605, Validation accuracy: 0.514
Epoch 9/20, Loss: 0.6827135852405003/0.6922801658511162, Validation accuracy: 0.514
Epoch 10/20, Loss: 0.6808079462202768/0.6931687705218792, Validation accuracy: 0.5165
Epoch 11/20, Loss: 0.6801090495926994/0.69356619566679, Validation accuracy: 0.52
Epoch 12/20, Loss: 0.6780784953208197/0.693211149